# Lab: Building an Interactive Dashboard

In the lecture, we built a dashboard one piece at a time. In this lab, you will bring together the analysis workflow, Plotly figures, and Dash components to create an interactive dashboard using a dataset of your choosing.

Your dashboard should help a specific audience answer a focused question. The goal is not to fit every possible chart onto one page. Select a small set of related evidence, organize it clearly, and give the user one useful way to interact with it.



## Learning goals

By the end of this lab, you should be able to:

- use the reproducible analysis workflow to plan a dashboard;
- prepare and check data before displaying it;
- create Plotly figures that answer related questions;
- organize text, controls, and figures in a Dash layout;
- connect an interactive control to figures with a callback; and
- explain the conclusion and limitations of a dashboard.



## Using AI tools responsibly

You may use an AI tool to ask questions, debug code, or receive feedback. You remain responsible for understanding every part of the submitted work, testing the dashboard, citing the dataset, and checking that the evidence supports your conclusion. Do not submit private, confidential, or personally identifying data to an AI system.



## Assignment requirements

Choose a dataset that supports a meaningful comparison. It should contain:

- enough observations to reveal a pattern rather than a few isolated cases;
- at least one categorical feature that can be used as an interactive filter or comparison;
- at least two additional useful features, which may be categorical, quantitative, temporal, spatial, or text-derived; and
- enough documentation for you to explain what the records and selected features represent.

The completed dashboard must contain:

- a descriptive title and a short explanation of its purpose;
- at least two related Plotly figures;
- at least one dropdown menu or radio-button control;
- at least one callback that updates both figures;
- human-readable titles, labels, categories, and colors;
- stable axes or category orders when changing them would make comparisons misleading; and
- a brief conclusion and limitation displayed on the page.



# Step 1: Question



## 1. Identify the audience

A dashboard should be designed for someone. Identify a realistic audience and explain what decision, judgment, or understanding the dashboard could support.



**Who is the intended audience, and why would this evidence be useful to them?**



The intended audience is emergency management professionals, weather researchers, and local government officials. This dashboard would help them understand how tornado activity varies across different states and months. By examining tornado frequency and intensity, they can identify areas and times with higher reported activity. This information could help support emergency preparedness, public awareness, and planning decisions.


## 2. State the research question

Write one focused question that your two figures will help answer. The question should name the main features or groups being compared.



**What question will the dashboard answer?**



How did the number and intensity of reported tornadoes vary across U.S. states and months during 2024?


## 3. Make a prediction



**What pattern do you expect to find, and what informed that prediction?**



I predict that tornado activity will be higher during the spring and early summer months, particularly in states such as Oklahoma, Texas, Kansas, and Iowa. I also expect weaker tornadoes to occur more frequently than stronger tornadoes. This prediction is based on my previous NOAA tornado investigation, which showed higher tornado counts in several central U.S. states during April and May 2024, along with general knowledge of seasonal tornado patterns.


# Step 2: Data



## 4. Import the libraries

Import Pandas, Plotly Express, and the Dash objects used in the lecture: `Dash`, `dcc`, `html`, `Input`, and `Output`.



In [2]:
%pip install dash

   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   -------------------------------------- - 8.7/8.9 MB 45.7 MB/s eta 0:00:01
   ---------------------------------------- 8.9/8.9 MB 29.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 41.7 MB/s  0:00:00
   ---------------------------------------- 0.0/876.3 kB ? eta -:--:--
   ---------------------------------------- 876.3/876.3 kB 24.6 MB/s  0:00:00

   ----------------------------------------  0/23 [zipp]
   ----------------------------------------  0/23 [zipp]
   - --------------------------------------  1/23 [urllib3]
   - --------------------------------------  1/23 [urllib3]
   - --------------------------------------  1/23 [urllib3]
   - --------------------------------------  1/23 [urllib3]
   - --------------------------------------  1/23 [urllib3]
   - --------------------------------------  1/23 [urllib3]
   - --------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output



## 5. Document the dataset

Give the dataset title, its creator or publisher, a working link or citation, the time period represented, and any relevant collection or sampling information.



**Where did the data come from, and what does its documentation tell you about how it was collected?**



The dataset is the NOAA Storm Events Database: 2024 Storm Events Details, published by the National Oceanic and Atmospheric Administration (NOAA) through the National Centers for Environmental Information (NCEI).

Dataset source:
https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2024_c20260728.csv.gz

Data documentation:
https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/Storm-Data-Bulk-csv-Format.pdf

The dataset covers reported storm events throughout the United States from January 1 through December 31, 2024. NOAA compiles these records using information collected by National Weather Service offices, including storm reports, observations, and damage assessments.

For this dashboard, I will use only records classified as tornadoes. Each record contains information about a reported tornado event, including its location, date, state, and Enhanced Fujita (EF) scale rating.

The dataset allows me to compare tornado activity across states and months while examining differences in reported tornado intensity. The information represents documented storm events rather than a random sample of weather conditions.


## 6. Read the data

Read the dataset into a Pandas DataFrame. Give the DataFrame a short, descriptive name.



In [5]:
# Load the NOAA 2024 Storm Events dataset
tornado_df = pd.read_csv(
    "data/StormEvents_details-ftp_v1.0_d2024_c20260728.csv",
    low_memory=False
)

# Keep only tornado events
tornado_df = tornado_df[
    tornado_df["EVENT_TYPE"] == "Tornado"
].copy()

# Convert event dates
tornado_df["BEGIN_DATE_TIME"] = pd.to_datetime(
    tornado_df["BEGIN_DATE_TIME"],
    format="mixed"
)

# Confirm the dataset loaded
print("Tornado dataset loaded successfully!")
print("Number of records:", len(tornado_df))
print("Number of columns:", len(tornado_df.columns))

Tornado dataset loaded successfully!
Number of records: 2137
Number of columns: 51


## 7. Preview and inspect the DataFrame

Display the first five rows, last five rows, stored data types, and missing-value counts.



In [6]:

# Display the first five rows
print("First five tornado records:")
display(tornado_df.head())

# Display the last five rows
print("Last five tornado records:")
display(tornado_df.tail())

# Display data types and non-missing values
print("Dataset information:")
tornado_df.info()

# Count missing values in every column
print("\nMissing values by column:")
display(tornado_df.isna().sum())


First five tornado records:


,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
19,202411,5,1101,202411,5,1102,196653,1216653,IOWA,19,...,4.10,ESE,BELKNAP,40.8030,-92.3482,40.8062,-92.3437,A brief tornado occurred with a shower in Davi...,A brief tornado occurred south of Floris from ...,CSV
107,202402,27,1932,202402,27,1935,188190,1155977,ILLINOIS,17,...,2.11,S,INVERNESS,42.0572,-88.1184,42.0898,-88.0937,"Following summer-like, record warmth, severe t...",This tornado touched down in the area near Del...,CSV
108,202402,27,1934,202402,27,1937,188190,1155978,ILLINOIS,17,...,1.69,SSW,INVERNESS,42.0722,-88.1485,42.0994,-88.1177,"Following summer-like, record warmth, severe t...",This tornado formed near Studio Drive in South...,CSV
113,202402,27,1815,202402,27,1816,188190,1155954,ILLINOIS,17,...,1.20,W,PAW PAW AIRPORT,41.7168,-89.0537,41.7173,-89.0529,"Following summer-like, record warmth, severe t...","A brief tornado, which was sighted by storm ch...",CSV
120,202402,27,1917,202402,27,1919,188190,1155975,ILLINOIS,17,...,0.31,SE,GENEVA,41.8827,-88.3482,41.8866,-88.3262,"Following summer-like, record warmth, severe t...",This tornado touched down along Keslinger Road...,CSV


Last five tornado records:


,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
69748,202406,26,1826,202406,26,1827,193907,1198110,PENNSYLVANIA,42,...,2.0,E,POPOFFS CORNERS,40.4269,-80.3060,40.4260,-80.2970,A well-organized upper shortwave trough crossi...,A National Weather Service survey concluded th...,CSV
69750,202406,5,2337,202406,5,2340,192638,1189052,OHIO,39,...,0.0,E,FRAYSZEBURG,40.1191,-82.1367,40.1194,-82.1131,Placeholder.,A strong tornado occurred during the overnight...,CSV
69752,202406,26,1906,202406,26,1920,193907,1198138,PENNSYLVANIA,42,...,1.0,NW,GREENWALD,40.3955,-79.6606,40.3768,-79.4560,A well-organized upper shortwave trough crossi...,"On the evening of Wednesday, June 26th a small...",CSV
69757,202406,26,1927,202406,26,1929,193907,1198121,PENNSYLVANIA,42,...,2.0,S,KIRBY,39.7769,-80.1302,39.7768,-80.1260,A well-organized upper shortwave trough crossi...,NWS Pittsburgh confirmed that an EF-0 tornado ...,CSV
69761,202406,26,1900,202406,26,1911,193907,1198137,PENNSYLVANIA,42,...,1.0,NE,HARRISON CITY,40.3790,-79.7960,40.3565,-79.6204,A well-organized upper shortwave trough crossi...,"On the evening of Wednesday, June 26th a small...",CSV


Dataset information:
<class 'pandas.DataFrame'>
Index: 2137 entries, 19 to 69761
Data columns (total 51 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   BEGIN_YEARMONTH     2137 non-null   int64         
 1   BEGIN_DAY           2137 non-null   int64         
 2   BEGIN_TIME          2137 non-null   int64         
 3   END_YEARMONTH       2137 non-null   int64         
 4   END_DAY             2137 non-null   int64         
 5   END_TIME            2137 non-null   int64         
 6   EPISODE_ID          2137 non-null   int64         
 7   EVENT_ID            2137 non-null   int64         
 8   STATE               2137 non-null   str           
 9   STATE_FIPS          2137 non-null   int64         
 10  YEAR                2137 non-null   int64         
 11  MONTH_NAME          2137 non-null   str           
 12  EVENT_TYPE          2137 non-null   str           
 13  CZ_TYPE             2137 non-null   str  

BEGIN_YEARMONTH          0
BEGIN_DAY                0
BEGIN_TIME               0
END_YEARMONTH            0
END_DAY                  0
END_TIME                 0
EPISODE_ID               0
EVENT_ID                 0
STATE                    0
STATE_FIPS               0
YEAR                     0
MONTH_NAME               0
EVENT_TYPE               0
CZ_TYPE                  0
CZ_FIPS                  0
CZ_NAME                  0
WFO                      0
BEGIN_DATE_TIME          0
CZ_TIMEZONE              0
END_DATE_TIME            0
INJURIES_DIRECT          0
INJURIES_INDIRECT        0
DEATHS_DIRECT            0
DEATHS_INDIRECT          0
DAMAGE_PROPERTY        363
DAMAGE_CROPS           358
SOURCE                   0
MAGNITUDE             2137
MAGNITUDE_TYPE        2137
FLOOD_CAUSE           2137
CATEGORY              2137
TOR_F_SCALE              0
TOR_LENGTH               0
TOR_WIDTH                0
TOR_OTHER_WFO         1786
TOR_OTHER_CZ_STATE    1786
TOR_OTHER_CZ_FIPS     1786
T

**Complete the table for every feature you expect to use. Add or remove rows as needed.**

| Feature           | What it represents               | Data storage type | Feature type | Missing values | Dashboard role         |
| ----------------- | -------------------------------- | ----------------- | ------------ | -------------: | ---------------------- |
| `STATE`           | State where the tornado occurred | `str`             | Categorical  |              0 | Dropdown filter        |
| `BEGIN_DATE_TIME` | Date and time the tornado began  | `datetime64[us]`  | Temporal     |              0 | Monthly comparisons    |
| `BEGIN_YEARMONTH` | Year and month of the event      | `int64`           | Temporal     |              0 | Monthly grouping       |
| `TOR_F_SCALE`     | Tornado intensity rating         | `str`             | Categorical  |              0 | Intensity comparison   |
| `EVENT_TYPE`      | Type of storm event              | `str`             | Categorical  |              0 | Filter tornado records |





In [8]:
# Inspect the features used in the dashboard
features = [
    "STATE",
    "BEGIN_DATE_TIME",
    "BEGIN_YEARMONTH",
    "TOR_F_SCALE",
    "EVENT_TYPE"
]

feature_summary = pd.DataFrame({
    "Data storage type": tornado_df[features].dtypes.astype(str),
    "Missing values": tornado_df[features].isna().sum()
})

display(feature_summary)

,Data storage type,Missing values
STATE,str,0
BEGIN_DATE_TIME,datetime64[us],0
BEGIN_YEARMONTH,int64,0
TOR_F_SCALE,str,0
EVENT_TYPE,str,0


**What does one row of the original dataset represent?**



Each row in the original NOAA Storm Events dataset represents one documented storm event reported in the United States during 2024. The original dataset includes different types of severe weather events, but I filtered it to include only tornadoes. After filtering, each row represents one reported tornado event, including its date, location, state, and intensity rating.


# Step 3: Operation



## 8. Create the analysis DataFrame

Select the features needed for the dashboard and use `.copy()` to create a DataFrame named `analysis`. Handle missing values in a way that is appropriate for the question. If values need clearer labels or corrected storage types, make those changes here.



In [9]:

# Select the features needed for the tornado dashboard
analysis = tornado_df[
    [
        "STATE",
        "BEGIN_DATE_TIME",
        "BEGIN_YEARMONTH",
        "TOR_F_SCALE",
        "EVENT_TYPE"
    ]
].copy()

# Ensure only tornado events are included
analysis = analysis[
    analysis["EVENT_TYPE"] == "Tornado"
].copy()

# Remove records missing essential dashboard information
analysis = analysis.dropna(
    subset=[
        "STATE",
        "BEGIN_DATE_TIME",
        "BEGIN_YEARMONTH",
        "TOR_F_SCALE"
    ]
)

# Display the number of records before and after preparation
print("Original tornado records:", len(tornado_df))
print("Prepared analysis records:", len(analysis))
print("Records removed:", len(tornado_df) - len(analysis))

# Preview the prepared dataset
display(analysis.head())


Original tornado records: 2137
Prepared analysis records: 2137
Records removed: 0


,STATE,BEGIN_DATE_TIME,BEGIN_YEARMONTH,TOR_F_SCALE,EVENT_TYPE
19,IOWA,2024-11-05 11:01:00,202411,EF0,Tornado
107,ILLINOIS,2024-02-27 19:32:00,202402,EF1,Tornado
108,ILLINOIS,2024-02-27 19:34:00,202402,EF1,Tornado
113,ILLINOIS,2024-02-27 18:15:00,202402,EF0,Tornado
120,ILLINOIS,2024-02-27 19:17:00,202402,EF1,Tornado


**How did you handle missing values, and why was that choice appropriate?**



I created a separate DataFrame named analysis containing the five features needed for my tornado dashboard. I kept only tornado events and checked for missing values in the state, date, month, and intensity fields. My earlier inspection showed that these features had no missing values, so I did not expect any records to be removed. This approach ensures that the dashboard uses complete records for comparing tornado frequency and intensity across states and months without changing the original DataFrame.


## 9. Create any calculated features

If the dashboard needs a calculated category, date component, percentage, rate, or other calculated feature, create it here. If no calculated feature is needed, leave the code cell blank and explain why.



In [12]:
# Create a readable month feature from the tornado event date
analysis["Month"] = analysis["BEGIN_DATE_TIME"].dt.month_name()

# Create a month number to keep months in chronological order
analysis["Month_Number"] = analysis["BEGIN_DATE_TIME"].dt.month

# Display the new calculated features
display(
    analysis[
        ["STATE", "BEGIN_DATE_TIME", "Month", "Month_Number", "TOR_F_SCALE"]
    ].head()
)

,STATE,BEGIN_DATE_TIME,Month,Month_Number,TOR_F_SCALE
19,IOWA,2024-11-05 11:01:00,November,11,EF0
107,ILLINOIS,2024-02-27 19:32:00,February,2,EF1
108,ILLINOIS,2024-02-27 19:34:00,February,2,EF1
113,ILLINOIS,2024-02-27 18:15:00,February,2,EF0
120,ILLINOIS,2024-02-27 19:17:00,February,2,EF1


**What calculated feature or cleaned label did you create, and how will it support the dashboard?**



I created two calculated features called Month and Month_Number using the tornado event dates. Month converts each date into a readable month name, while Month_Number identifies the month numerically from 1 to 12. These features will help organize tornado activity chronologically and make it easier to compare the number of reported tornadoes across different months and states during 2024.


## 10. Prepare the control options

Create the labels and values for the dropdown or radio-button control. The visible labels should be understandable to someone who has not seen the original column names.



In [13]:
# Create a sorted list of states in the dataset
state_list = sorted(analysis["STATE"].unique())

# Create dropdown options with readable labels
state_options = [
    {"label": "All States", "value": "All States"}
] + [
    {"label": state.title(), "value": state}
    for state in state_list
]

# Set the default selection
default_state = "All States"

# Display the dropdown options
print("Number of states and territories:", len(state_list))
print("Total dropdown options:", len(state_options))

display(state_options)

Number of states and territories: 47
Total dropdown options: 48


[{'label': 'All States', 'value': 'All States'},
 {'label': 'Alabama', 'value': 'ALABAMA'},
 {'label': 'Alaska', 'value': 'ALASKA'},
 {'label': 'Arizona', 'value': 'ARIZONA'},
 {'label': 'Arkansas', 'value': 'ARKANSAS'},
 {'label': 'California', 'value': 'CALIFORNIA'},
 {'label': 'Colorado', 'value': 'COLORADO'},
 {'label': 'Connecticut', 'value': 'CONNECTICUT'},
 {'label': 'Delaware', 'value': 'DELAWARE'},
 {'label': 'Florida', 'value': 'FLORIDA'},
 {'label': 'Georgia', 'value': 'GEORGIA'},
 {'label': 'Idaho', 'value': 'IDAHO'},
 {'label': 'Illinois', 'value': 'ILLINOIS'},
 {'label': 'Indiana', 'value': 'INDIANA'},
 {'label': 'Iowa', 'value': 'IOWA'},
 {'label': 'Kansas', 'value': 'KANSAS'},
 {'label': 'Kentucky', 'value': 'KENTUCKY'},
 {'label': 'Louisiana', 'value': 'LOUISIANA'},
 {'label': 'Maryland', 'value': 'MARYLAND'},
 {'label': 'Massachusetts', 'value': 'MASSACHUSETTS'},
 {'label': 'Michigan', 'value': 'MICHIGAN'},
 {'label': 'Minnesota', 'value': 'MINNESOTA'},
 {'label': 'Mi

# Step 4: Check



## 11. Check the prepared data

Display the first rows of `analysis`, its stored data types, missing-value counts, and appropriate summaries or frequency counts for the dashboard features.



In [14]:
# Display the first five rows of the prepared dataset
print("First five rows of the analysis DataFrame:")
display(analysis.head())

# Check the data types
print("\nData types:")
display(analysis.dtypes)

# Check for missing values
print("\nMissing values:")
display(analysis.isna().sum())

# Display the total number of tornado records
print("\nTotal tornado records:", len(analysis))

# Count tornadoes by state
print("\nTornado counts by state:")
display(analysis["STATE"].value_counts())

# Count tornadoes by month in chronological order
print("\nTornado counts by month:")
monthly_counts = (
    analysis.groupby(["Month_Number", "Month"])
    .size()
    .reset_index(name="Tornado_Count")
    .sort_values("Month_Number")
)

display(monthly_counts)

# Count tornadoes by intensity rating
print("\nTornado counts by EF scale:")
display(analysis["TOR_F_SCALE"].value_counts().sort_index())



First five rows of the analysis DataFrame:


,STATE,BEGIN_DATE_TIME,BEGIN_YEARMONTH,TOR_F_SCALE,EVENT_TYPE,Month,Month_Number
19,IOWA,2024-11-05 11:01:00,202411,EF0,Tornado,November,11
107,ILLINOIS,2024-02-27 19:32:00,202402,EF1,Tornado,February,2
108,ILLINOIS,2024-02-27 19:34:00,202402,EF1,Tornado,February,2
113,ILLINOIS,2024-02-27 18:15:00,202402,EF0,Tornado,February,2
120,ILLINOIS,2024-02-27 19:17:00,202402,EF1,Tornado,February,2



Data types:


STATE                         str
BEGIN_DATE_TIME    datetime64[us]
BEGIN_YEARMONTH             int64
TOR_F_SCALE                   str
EVENT_TYPE                    str
Month                         str
Month_Number                int32
dtype: object


Missing values:


...

STATE
OKLAHOMA          178
ILLINOIS          174
IOWA              155
TEXAS             149
FLORIDA           132
MISSOURI          119
NEBRASKA          115
LOUISIANA         108
MISSISSIPPI        94
OHIO               90
ALABAMA            80
INDIANA            73
KANSAS             69
KENTUCKY           66
ARKANSAS           62
SOUTH CAROLINA     57
WISCONSIN          55
NORTH CAROLINA     49
GEORGIA            38
MINNESOTA          34
NEW YORK           33
PENNSYLVANIA       24
TENNESSEE          23
WEST VIRGINIA      21
COLORADO           21
MICHIGAN           18
VIRGINIA           14
NORTH DAKOTA       14
MARYLAND           13
NEW MEXICO         12
ARIZONA             7
SOUTH DAKOTA        7
WYOMING             5
CALIFORNIA          4
WASHINGTON          4
MONTANA             4
OREGON              3
MASSACHUSETTS       2
NEW HAMPSHIRE       2
UTAH                2
CONNECTICUT         1
NEW JERSEY          1
IDAHO               1
ALASKA              1
RHODE ISLAND        1
DELA


Tornado counts by month:


,Month_Number,Month,Tornado_Count
0,1,January,52
1,2,February,54
2,3,March,77
3,4,April,413
4,5,May,655
5,6,June,170
6,7,July,253
7,8,August,95
8,9,September,76
9,10,October,66



Tornado counts by EF scale:


TOR_F_SCALE
EF0    698
EF1    923
EF2    181
EF3     51
EF4      5
EFU    279
Name: count, dtype: int64

## 12. Check the interactive groups

Calculate the number of observations available for every control option. Make sure each option returns data and that unexpectedly small groups are understood before building the callback.



In [15]:
# Count tornado records for each state
state_counts = (
    analysis["STATE"]
    .value_counts()
    .sort_index()
)

# Include the All States dropdown option
control_counts = pd.concat([
    pd.Series({"All States": len(analysis)}),
    state_counts
])

# Display the number of records for every dropdown option
print("Number of tornado records for each dropdown option:")
display(control_counts.to_frame(name="Tornado_Count"))

# Verify that every dropdown option has data
print("\nDropdown validation:")

for option in state_options:
    state = option["value"]

    if state == "All States":
        count = len(analysis)
    else:
        count = (analysis["STATE"] == state).sum()

    if count == 0:
        print(f"WARNING: {option['label']} has no tornado records!")

print("\nAll dropdown options checked.")

# Identify states with fewer than 10 tornado records
print("\nStates with fewer than 10 tornado records:")
display(
    state_counts[state_counts < 10]
    .to_frame(name="Tornado_Count")
)

# Display the number of states with tornado records
print("\nTotal states and territories represented:", len(state_counts))



Number of tornado records for each dropdown option:


,Tornado_Count
All States,2137
ALABAMA,80
ALASKA,1
ARIZONA,7
ARKANSAS,62
CALIFORNIA,4
COLORADO,21
CONNECTICUT,1
DELAWARE,1
FLORIDA,132



Dropdown validation:

All dropdown options checked.

States with fewer than 10 tornado records:


,Tornado_Count
STATE,
ALASKA,1
ARIZONA,7
CALIFORNIA,4
CONNECTICUT,1
DELAWARE,1
IDAHO,1
MASSACHUSETTS,2
MONTANA,4
NEW HAMPSHIRE,2



Total states and territories represented: 47


**What did these checks establish about the data that will appear in the dashboard?**



These checks establish how many tornado records are available for each state in the interactive dropdown menu. The All States option includes the complete dataset of 2,137 reported tornadoes, while individual state selections display only records from that state. Checking the number of observations helps identify states with relatively few tornado reports, where patterns may be less reliable. The validation also helps confirm that each dropdown option contains data before the dashboard is built.


# Step 5: Evidence



## 13. Create the first static Plotly figure

Build the first figure before placing it in Dash. Give it a descriptive title and human-readable axis and legend labels.



In [16]:
# Define the correct chronological order of months
month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

# Count tornadoes for each month
monthly_tornadoes = (
    analysis.groupby("Month")
    .size()
    .reindex(month_order, fill_value=0)
    .reset_index(name="Tornado_Count")
)

# Create the first static Plotly figure
fig1 = px.bar(
    monthly_tornadoes,
    x="Month",
    y="Tornado_Count",
    title="Reported U.S. Tornadoes by Month — 2024",
    labels={
        "Month": "Month",
        "Tornado_Count": "Number of Reported Tornadoes"
    },
    color="Tornado_Count",
    color_continuous_scale="Blues"
)

# Improve readability
fig1.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Reported Tornadoes",
    xaxis_tickangle=-45,
    coloraxis_showscale=False,
    template="plotly_white"
)

# Display the figure
fig1.show()



**What evidence should a viewer obtain from the first figure?**



The first figure shows how the number of reported tornadoes changed throughout the United States during 2024. By comparing monthly tornado counts, viewers can identify which months experienced the highest and lowest tornado activity. This evidence helps determine whether tornadoes were more common during the spring and early summer months, as predicted. It also provides a nationwide overview that can later be compared with individual states using the dashboard's interactive dropdown menu.


## 14. Create the second static Plotly figure

The second figure should add different but related evidence rather than repeat the first figure in another style.



In [17]:
# Define the order of tornado intensity ratings
ef_order = ["EF0", "EF1", "EF2", "EF3", "EF4", "EF5", "EFU"]

# Count tornadoes by EF scale rating
intensity_counts = (
    analysis["TOR_F_SCALE"]
    .value_counts()
    .reindex(ef_order, fill_value=0)
    .reset_index()
)

# Rename columns for readability
intensity_counts.columns = ["EF_Rating", "Tornado_Count"]

# Create the second static Plotly figure
fig2 = px.bar(
    intensity_counts,
    x="EF_Rating",
    y="Tornado_Count",
    title="Reported U.S. Tornadoes by Intensity — 2024",
    labels={
        "EF_Rating": "Enhanced Fujita (EF) Scale Rating",
        "Tornado_Count": "Number of Reported Tornadoes"
    },
    color="EF_Rating",
    category_orders={"EF_Rating": ef_order},
    color_discrete_sequence=[
        "#90CAF9",
        "#42A5F5",
        "#1E88E5",
        "#1565C0",
        "#0D47A1",
        "#082567",
        "#9E9E9E"
    ]
)

# Improve chart readability
fig2.update_layout(
    xaxis_title="Tornado Intensity (EF Scale)",
    yaxis_title="Number of Reported Tornadoes",
    template="plotly_white",
    showlegend=False
)

# Display the figure
fig2.show()

# Display the exact intensity counts
display(intensity_counts)



,EF_Rating,Tornado_Count
0,EF0,698
1,EF1,923
2,EF2,181
3,EF3,51
4,EF4,5
5,EF5,0
6,EFU,279


**What does the second figure add to the first figure's evidence?**



The second figure adds information about tornado intensity by showing how many reported tornadoes fell into each Enhanced Fujita (EF) scale category during 2024. While the first figure compares tornado frequency across months, the second figure compares the strength of those tornadoes. This allows viewers to determine whether weaker tornadoes were reported more frequently than stronger tornadoes. Together, the two figures provide a more complete understanding of tornado activity by examining both when tornadoes occurred and how intense they were.


## 15. Plan the dashboard



**Complete the dashboard plan before writing the app. Add rows if needed.**

| Component                     | What it displays or controls                                                                                                                                 | Why it belongs on the dashboard                                                                                                  |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------------------------------- |
| **Title and introduction**    | Displays the title "2024 U.S. Tornado Activity Dashboard" and explains that the dashboard compares tornado frequency and intensity across states and months. | Helps emergency management professionals, weather researchers, and local officials understand the dashboard's purpose.           |
| **Interactive control**       | A dropdown menu that allows users to select an individual U.S. state or view all states together.                                                            | Allows viewers to compare tornado activity in specific states with nationwide patterns.                                          |
| **First figure**              | A bar chart showing the number of reported tornadoes for each month from January through December 2024.                                                      | Identifies seasonal patterns and shows which months experienced the highest and lowest tornado activity.                         |
| **Second figure**             | A bar chart showing the number of reported tornadoes by Enhanced Fujita (EF) scale rating, including unknown ratings.                                        | Shows the distribution of tornado intensity and helps viewers compare weaker and stronger tornadoes.                             |
| **Conclusion and limitation** | A brief summary of tornado activity during 2024 and a statement explaining that the dashboard uses reported tornado events from one year.                    | Helps viewers interpret the findings while recognizing that one year of reported data cannot establish long-term tornado trends. |




## 16. Create reusable figure code

Write a function that accepts the selected control value, filters or summarizes `analysis`, creates both Plotly figures, and returns the figures in a consistent order. Fix axis ranges or category orders when they need to remain stable across selections.



In [20]:
# Create a reusable function for both dashboard figures
def create_dashboard_figures(selected_state):

    # Filter the dataset based on the selected state
    if selected_state == "All States":
        filtered_data = analysis.copy()
        location_title = "United States"
    else:
        filtered_data = analysis[
            analysis["STATE"] == selected_state
        ].copy()
        location_title = selected_state.title()

    # FIGURE 1: Monthly Tornado Activity

    # Count tornadoes by month
    monthly_data = (
        filtered_data.groupby("Month")
        .size()
        .reindex(month_order, fill_value=0)
        .reset_index(name="Tornado_Count")
    )

    # Create monthly tornado bar chart
    monthly_fig = px.bar(
        monthly_data,
        x="Month",
        y="Tornado_Count",
        title=f"Reported Tornadoes by Month — {location_title} (2024)",
        labels={
            "Month": "Month",
            "Tornado_Count": "Number of Reported Tornadoes"
        },
        color="Tornado_Count",
        color_continuous_scale="Blues"
    )

    monthly_fig.update_layout(
        xaxis_title="Month",
        yaxis_title="Number of Reported Tornadoes",
        xaxis_tickangle=-45,
        coloraxis_showscale=False,
        template="plotly_white"
    )

    # Keep month order and axis range consistent
    monthly_fig.update_xaxes(
        categoryorder="array",
        categoryarray=month_order
    )

    monthly_fig.update_yaxes(
        range=[0, monthly_tornadoes["Tornado_Count"].max() * 1.1]
    )

    # FIGURE 2: Tornado Intensity Distribution

    # Count tornadoes by EF-scale rating
    intensity_data = (
        filtered_data["TOR_F_SCALE"]
        .value_counts()
        .reindex(ef_order, fill_value=0)
        .reset_index()
    )

    intensity_data.columns = ["EF_Rating", "Tornado_Count"]

    # Create intensity bar chart
    intensity_fig = px.bar(
        intensity_data,
        x="EF_Rating",
        y="Tornado_Count",
        title=f"Reported Tornadoes by Intensity — {location_title} (2024)",
        labels={
            "EF_Rating": "Enhanced Fujita (EF) Scale Rating",
            "Tornado_Count": "Number of Reported Tornadoes"
        },
        color="EF_Rating",
        category_orders={"EF_Rating": ef_order},
        color_discrete_sequence=[
            "#90CAF9",
            "#42A5F5",
            "#1E88E5",
            "#1565C0",
            "#0D47A1",
            "#082567",
            "#9E9E9E"
        ]
    )

    intensity_fig.update_layout(
        xaxis_title="Tornado Intensity (EF Scale)",
        yaxis_title="Number of Reported Tornadoes",
        template="plotly_white",
        showlegend=False
    )

    # Keep intensity axis range consistent
    intensity_fig.update_yaxes(
        range=[0, intensity_counts["Tornado_Count"].max() * 1.1]
    )

    # Return both figures in the same order
    return monthly_fig, intensity_fig


# Confirm that the function was created successfully
print("Reusable dashboard figure function created successfully!")

Reusable dashboard figure function created successfully!


## 17. Build the layout

Create a Dash app and its layout. Include the title, purpose statement, interactive control, two empty `dcc.Graph` components with unique IDs, and short conclusion and limitation statements.



In [21]:
# Create the Dash application
app = Dash(__name__)

# Build the dashboard layout
app.layout = html.Div(

    children=[

        # Dashboard title
        html.H1(
            "2024 U.S. Tornado Activity Dashboard",
            style={
                "textAlign": "center",
                "color": "#1E3A5F"
            }
        ),

        # Dashboard introduction
        html.P(
            "Explore how the frequency and intensity of reported "
            "tornadoes varied across U.S. states and months during 2024. "
            "Select a state to compare its tornado activity with "
            "nationwide patterns.",
            style={
                "textAlign": "center",
                "fontSize": "18px",
                "marginBottom": "30px"
            }
        ),

        # State selection dropdown
        html.Div(
            children=[

                html.Label(
                    "Select a State:",
                    style={
                        "fontWeight": "bold",
                        "fontSize": "18px"
                    }
                ),

                dcc.Dropdown(
                    id="state-dropdown",
                    options=state_options,
                    value=default_state,
                    clearable=False
                )

            ],
            style={
                "width": "50%",
                "margin": "auto",
                "marginBottom": "30px"
            }
        ),

        # First graph: Monthly tornado activity
        dcc.Graph(
            id="monthly-tornado-graph"
        ),

        # Second graph: Tornado intensity distribution
        dcc.Graph(
            id="tornado-intensity-graph"
        ),

        # Dashboard conclusion
        html.H3(
            "Conclusion",
            style={"color": "#1E3A5F"}
        ),

        html.P(
            "This dashboard compares the monthly frequency and "
            "reported intensity of tornadoes during 2024. "
            "The charts help identify seasonal patterns and differences "
            "in tornado activity between states."
        ),

        # Dashboard limitation
        html.H3(
            "Limitation",
            style={"color": "#1E3A5F"}
        ),

        html.P(
            "The dashboard uses reported tornado events from 2024 only. "
            "Reporting practices and incomplete intensity assessments "
            "may affect the results. One year of data cannot establish "
            "long-term tornado trends."
        )

    ],

    style={
        "maxWidth": "1200px",
        "margin": "auto",
        "padding": "25px",
        "fontFamily": "Arial, sans-serif",
        "backgroundColor": "#F8FAFC"
    }
)

print("Dash dashboard layout created successfully!")



Dash dashboard layout created successfully!


## 18. Connect the callback

Connect the control's `value` to the `figure` property of both graphs. The callback function should call the reusable figure function and return its two figures in the same order as the two outputs.



In [22]:
# Connect the state dropdown to both dashboard graphs
@app.callback(
    Output("monthly-tornado-graph", "figure"),
    Output("tornado-intensity-graph", "figure"),
    Input("state-dropdown", "value")
)
def update_dashboard(selected_state):

    # Generate both figures using the selected state
    monthly_fig, intensity_fig = create_dashboard_figures(
        selected_state
    )

    # Return both updated figures
    return monthly_fig, intensity_fig


print("Dashboard callback connected successfully!")



Dashboard callback connected successfully!


## 19. Check the callback as a Python function

Call the callback or reusable figure function with at least two valid control values. Confirm that each call returns two Plotly figures without an error.



In [25]:
import plotly.graph_objects as go

# Test the dashboard with two dropdown selections
test_states = ["All States", "OKLAHOMA"]

for state in test_states:
    print(f"\nTesting dashboard selection: {state}")

    monthly_fig, intensity_fig = create_dashboard_figures(state)

    assert isinstance(monthly_fig, go.Figure)
    assert isinstance(intensity_fig, go.Figure)

    print("Monthly tornado figure: PASS")
    print("Tornado intensity figure: PASS")
    print("Both figures generated successfully!")

print("\nAll dashboard function tests passed!")


Testing dashboard selection: All States
Monthly tornado figure: PASS
Tornado intensity figure: PASS
Both figures generated successfully!

Testing dashboard selection: OKLAHOMA
Monthly tornado figure: PASS
Tornado intensity figure: PASS
Both figures generated successfully!

All dashboard function tests passed!


## 20. Run the dashboard

Run the app in a browser tab. Use a port that is not already occupied by another app, and stop the app before restarting the same code.



In [24]:
# Run the interactive tornado dashboard
print("Starting the 2024 U.S. Tornado Activity Dashboard...")
print("Open http://127.0.0.1:8050/ in your browser.")

app.run(
    host="127.0.0.1",
    port=8050,
    debug=False,
    use_reloader=False
)



Starting the 2024 U.S. Tornado Activity Dashboard...
Open http://127.0.0.1:8050/ in your browser.


**After testing every control option, what did you revise or confirm about the dashboard's behavior?**



After testing the dashboard's dropdown options, I confirmed that selecting a state updates both charts to display tornado records for that location. The All States option displays nationwide tornado activity, while individual state selections show the monthly frequency and intensity distribution for that state. The month labels remain in chronological order, and the EF-scale categories maintain a consistent arrangement. Both charts use fixed vertical axis ranges to support accurate comparisons between selections.


**Which design choice most improves the dashboard's readability or usefulness?**



The most useful design choice is the interactive state dropdown because it allows viewers to examine tornado activity for individual states without navigating between separate charts or pages. Both figures update together, making it easier to compare monthly tornado frequency and intensity for the same location. Keeping the months and EF-scale ratings in consistent orders also improves readability and makes comparisons between states more meaningful.


# Step 6: Conclusion



**State a concise conclusion that directly answers the research question and refers to evidence visible in the dashboard.**



The analysis of NOAA's 2024 Storm Events Database shows that reported tornado activity varied across U.S. states and throughout the year. The dashboard's monthly tornado chart allows viewers to identify seasonal differences in tornado frequency, while the intensity chart shows how tornado reports were distributed across Enhanced Fujita (EF) scale ratings. Together, these visualizations demonstrate that tornado activity was not evenly distributed across locations or months and provide a way to compare the frequency and reported intensity of tornadoes during 2024.


**How does the interactive control help the audience examine the evidence?**



The interactive state dropdown allows users to explore tornado activity across the United States by selecting individual states or viewing nationwide data. When a state is selected, both charts automatically update to display the monthly tornado frequency and intensity distribution for that location. This helps emergency management professionals, weather researchers, and local government officials identify differences in tornado activity between states. The consistent month ordering, EF-scale categories, and axis ranges make comparisons easier to interpret. By allowing users to focus on specific locations, the dashboard provides a more useful and accessible way to examine tornado patterns during 2024.


# Step 7: Limitation



**State one important limitation on how the dashboard's evidence should be interpreted.**



One important limitation of this dashboard is that it only examines reported tornado events from 2024. Although the data provides useful information about tornado frequency and intensity across different states and months, a single year cannot establish long-term weather patterns or trends. Additionally, the dataset relies on documented tornado reports, which may be affected by differences in reporting practices, available observations, and incomplete intensity assessments. Therefore, the dashboard should be used to understand reported tornado activity during 2024 rather than predict future tornado occurrences or determine long-term changes in tornado frequency.


## AI-use reflection



**Describe any AI assistance you used. Identify one suggestion you verified, revised, or rejected. If you did not use AI, state that.**



I used ChatGPT for coding assistance and troubleshooting. I revised its suggestion for validating Plotly figures to use a simpler method.


## Submission checklist

Before submitting, confirm that:

- [ ] every prompt has one complete response;
- [ ] all notebook cells run in order without an error;
- [ ] the dashboard opens and every control option works;
- [ ] the two figures use readable titles, labels, categories, and colors;
- [ ] comparable views retain stable axes or category orders when needed;
- [ ] the dataset source is cited and the required data file accompanies the submission;
- [ ] the notebook and dashboard code are organized and readable;
- [ ] the conclusion and limitation are supported by the displayed evidence; and
- [ ] the work is submitted by the deadline.

